# Extended Data Fig. 19c,d - Raw regional volume computation

This notebook computes the **raw-space regional volume** that underlies Extended Data Fig. 19c,d. For each brain it builds a regular (~16.7 um) grid over the brain mask in the original (raw) imaging space, then registers that grid to the Neuron Atlas / CUBIC / SCA spaces with ANTs. Counting grid points per atlas region gives the **raw regional volume** (volume measured in native space, before atlas-based volume normalization).

In Extended Data Fig. 19c,d this raw volume is compared against atlas-corrected volume for the claustrum (CLA): the raw regional volume shows a significant decrease in 9-month App<sup>NL-G-F</sup> mice, neuronal density computed with the raw volume is non-significant, whereas density computed with the atlas-corrected volume reveals a significant neuronal decrease.

> **Note.** This notebook covers the raw-volume *computation* step (one representative sample batch is shown; the identical routine was applied to the remaining WT/App batches). The downstream CLA comparison, statistics and plotting are performed in the main regional-analysis notebook. This data-generation step produces grid/volume files and has no figure output.

## Input files (per sample)

For each sample, paths are derived from its pipeline parameter file:

| Input | Description |
|---|---|
| `param/Neuronomics/<sample>/param_multichannel-rank.json` | gives `dst_basedir` for the sample |
| `<dst_basedir>/scalemerged_/img_density.tif` | cell-density image; binarized to a brain mask |
| `<...>/nu_R_/result/F2M_0GenericAffine.mat`, `F2M_InvWarped.nii.gz` | ANTs transforms (brain -> atlas) |
| `/data4/Neurology_project_data_archive/atlas_R_test/slice_0_88_inv.mat` | atlas slice transform |

Outputs (per sample) are raw/CUBIC/SCA-space grid `.bin` files and check images written next to the sample data.

In [ ]:
import os, re, json, gc, datetime
import numpy as np
import pandas as pd
import tifffile
import ants
from scipy import ndimage
from scipy.spatial import cKDTree

In [ ]:
import tifffile
from scipy import ndimage
import numpy as np

# For a new brain: first register to the R+ atlas; the transformed grid is df_transformed.
start_time = datetime.datetime.now()
print("Start time:", start_time)
#Samples = ["#5_APPmodel_Ctr9m_1_2023_0130_1607", ]#"#5_APPmodel_Ctr9m_2_2023_0216_2052" ,"#5_APPmodel_Ctr9m_3_2023_0322_2014","#5_APPmodel_Ctr9m_4_2023_0518_1449","#5_APPmodel_APP9m_1_2023_0117_1352","#5_APPmodel_APP9m_2_2023_0327_1425", "#5_APPmodel_APP9m_3_2023_0329_1611", "#5_APPmodel_APP9m_4_2023_0403_1443" ]
Samples = ["#4_APPmodel_APP1m_1_2022_1102_1304", "#5_APPmodel_APP3m_4_2023_0214_2027","#4_APPmodel_APP5m_4_2023_0509_1021","#5_APPmodel_APP7m_5_2023_0520_1723"  ]
for Sample in Samples:
    param_path = "/home/mitani/CUBIC-informatics/param/Neuronomics/" + Sample + "/param_multichannel-rank.json"

    #print(param_path)

    #try:

    with open(param_path) as f:
        param = json.load(f)

    fw_dir_nu = param["dst_basedir"]
    #######################################
    if ("Neurology_project_data_archive" not in fw_dir_nu and
        not re.search(r'/ds1_Imaging[23]/', fw_dir_nu)):

        fw_dir_nu = re.sub(
            r'^(/data\d+/)',                  # right after /dataX/
            r'\1Neurology_project_data_archive/',
            fw_dir_nu
        )
    ##########################################
    print(fw_dir_nu )
    
    fw_dir_nu_img_density = fw_dir_nu.replace("intensities_", "scalemerged_/img_density.tif")
    img_density = tifffile.imread(fw_dir_nu_img_density)
    print(img_density.shape)
    
    img_density[img_density>0] =1
    img_density[img_density<=0] =0
    img_density = ndimage.binary_fill_holes(img_density)
    tifffile.imsave(fw_dir_nu_img_density.replace("img_density", "img_density_binary_closed"), img_density.astype(np.uint16))
    # Get coordinates (and values) of non-zero voxels (the brain mask)
    non_zero_coords = np.argwhere(img_density)
    non_zero_values = img_density[non_zero_coords[:, 0], non_zero_coords[:, 1], non_zero_coords[:, 2]]

    # Convert to DataFrame
    df_non_zero = pd.DataFrame(non_zero_coords, columns=['z', 'y', 'x'])
    df_non_zero['id'] = non_zero_values   
    
    # Select / rename columns
    df_save = df_non_zero.copy()
    #df.columns = ['X(um)', 'Y(um)', 'Z(um)', 'atlasID']
    #df
    df_save.columns = ['X(um)', 'Y(um)', 'Z(um)', 'atlasID'] # raw xyz
    df_save["X(um)"] = df_non_zero ["x"] #* 50  # handled in 50um scale
    df_save["Y(um)"] = df_non_zero ["y"] #* 50
    df_save["Z(um)"] = df_non_zero ["z"] #* 50
    df_save["atlasID"] = df_non_zero ["id"]     
    #df_save.to_csv(fw_dir_nu + "50um_grid_on_raw_space.csv.gz", index=False, compression='gzip')
    
    df_save4 = df_save#pd.read_csv(fw_dir_nu + "50um_grid_on_raw_space.csv.gz", compression='gzip')
    
    # Build a regular grid over the brain mask
    
    if 1==1:
        print("IN")

        # step size
        z_step = 1 / 3  # ~16.7um spacing
        y_step = 1 / 3 #5
        x_step = 1 / 3 #5

        # full-size xy grid
        y_indices = np.arange(0, img_density.shape[1], y_step)
        x_indices = np.arange(0, img_density.shape[2], x_step)

        # xy-plane grid for a single z
        y_vals = np.repeat(y_indices, len(x_indices))
        x_vals = np.tile(x_indices, len(y_indices))
        z_vals = np.zeros(len(x_vals))  # initial z = 0

        # DataFrame for a single z-plane
        single_z_grid_df = pd.DataFrame({
            'z': z_vals,
            'y': y_vals,
            'x': x_vals
        })

        del z_vals,y_vals,x_vals
        gc.collect()

        # z values for the first block
        first_block_z_values = np.arange(0, img_density.shape[0] / 5, z_step)
        first_block_z_values = np.round(first_block_z_values, 3)

        # build the first block
        first_block_dfs = []
        for z in first_block_z_values:
            print(z)
            temp_df = single_z_grid_df.copy()
            temp_df['z'] = z
            first_block_dfs.append(temp_df)
        del temp_df, single_z_grid_df
        gc.collect()

        # concatenate into one DataFrame
        first_block_df = pd.concat(first_block_dfs, ignore_index=True)
        del first_block_dfs
        gc.collect()

        # initialize output DataFrame
        grid_points_df = pd.DataFrame(columns=['z', 'y', 'x'])

        tree = cKDTree(df_save4[['X(um)', 'Y(um)', 'Z(um)']].values)

        # repeat the first block 5x to cover the whole stack
        for i in range(5):
            print(i)
            temp_df = first_block_df.copy()
            # offset z for this block
            temp_df['z'] += i * (img_density.shape[0] / 5)
            # nearest-neighbour query against the mask points

            distances, indexes = tree.query(temp_df[['x', 'y', 'z']].values, k=1, distance_upper_bound = 2, n_jobs = -1)
            mask = distances <= 2 ** 0.5


            grid_points_df = pd.concat([grid_points_df, temp_df[mask]], ignore_index=True)

            print(str((len(temp_df[mask])/len(temp_df)) * 100) + "%")

            del temp_df, mask, distances, indexes
            gc.collect()

        # free memory
        del first_block_df
        gc.collect()

        # done
        print("Grid generation completed.")
        
        grid_points_df2 = grid_points_df.copy()
        grid_points_df2[['z', 'y', 'x']] = grid_points_df[['z', 'y', 'x']] * 50 # um scale
        
        # register grid points to the atlas (8w-1) space via ANTs
        if 1==1:
            df_transformed = ants.apply_transforms_to_points(
            dim=3,
            points=grid_points_df2,
            transformlist=[os.path.join(fw_dir_nu.replace("intensities_", "nu_R_/result"), "F2M_0GenericAffine.mat"),
                        os.path.join(fw_dir_nu.replace("intensities_", "nu_R_/result"), "F2M_InvWarped.nii.gz")],
            whichtoinvert = [True, False]
            )
                                
        df_transformed[['z', 'y', 'x']] = df_transformed[['z', 'y', 'x']] * 1/50 # 50um scale
        if 1==1:
            transformed_points = ants.apply_transforms_to_points(
                dim=3,
                points=df_transformed,
                transformlist=["/data4/Neurology_project_data_archive/atlas_R_test/slice_0_88_inv.mat"],
                whichtoinvert = [True]
            )

            print("trasform end")

        #del   grid_points_df
        #gc.collect()




    transformed_points = pd.DataFrame({'id': range(len(transformed_points)), 'x': transformed_points["x"], 'y': transformed_points["y"], 'z': transformed_points["z"]})

                                                         
    dt_stitched = np.dtype([
        ('z', 'f4'), ('y', 'f4'), ('x', 'f4'),
    ])

    data_annotated =  np.zeros((grid_points_df.shape[0],), dtype=dt_stitched)
    data_annotated["x"]  = grid_points_df["x"] # 50um scale
    data_annotated["y"]  = grid_points_df["y"]
    data_annotated["z"]  = grid_points_df["z"]
    data_annotated.tofile(fw_dir_nu + "Raw-space-grid-16.7um-resolution_ver1.bin")
    print(data_annotated)                                                 
                                                         
    # rasterize grid points to an image for visual check
    if 1==1:
        scale = 1/50

        depth_ori = img_density.shape[0]
        height_ori = img_density.shape[1]
        width_ori = img_density.shape[2]

        print(depth_ori)
        print(height_ori)
        print(width_ori)

        img_filename_Nuclear_Isocortex_to_ori = fw_dir_nu + "Raw_16.7um_grid_points_50um.tif"

        img_N_ori,_ = np.histogramdd(
            np.vstack([
                data_annotated["z"],
                data_annotated["y"], 
                data_annotated["x"] 
                ]).T,
                bins=(depth_ori, height_ori, width_ori),
                range=[(0,depth_ori),(0,height_ori),(0,width_ori)]
            )

        tifffile.imsave(
                img_filename_Nuclear_Isocortex_to_ori,
                img_N_ori.astype(np.float32)
            )
        del img_N_ori
        gc.collect()   
                                                         
    # save split into 6 parts
    if 1==1:
        df_list = []
        df_list = [grid_points_df.iloc[offset::6] for offset in range(6)]
        dt_stitched = np.dtype([
            ('z', 'f4'), ('y', 'f4'), ('x', 'f4')
        ])

        for i in [0, 1,2, 3, 4, 5]:
            data_annotated =  np.zeros((len(df_list[i]),), dtype=dt_stitched)
            data_annotated["x"]  = df_list[i]["x"] # 50um scale
            data_annotated["y"]  = df_list[i]["y"]
            data_annotated["z"]  = df_list[i]["z"]
            #data_annotated["id"]  = df_list[i]["id"]
            data_annotated.tofile(fw_dir_nu + "Raw-space-grid-16.7um-resolution_ver1_" +str(i)+".bin")   
                                                        
    dt_stitched = np.dtype([
        ('z', 'f4'), ('y', 'f4'), ('x', 'f4'),
    ])

    data_annotated =  np.zeros((df_transformed.shape[0],), dtype=dt_stitched)
    data_annotated["x"]  = df_transformed["x"] # 50um scale
    data_annotated["y"]  = df_transformed["y"]
    data_annotated["z"]  = df_transformed["z"]
    data_annotated.tofile(fw_dir_nu + "CUBIC-space-grid-16.7um-resolution_ver1.bin")
    print(data_annotated)                                                      
                                                         
    # rasterize grid points to an image for visual check
    if 1==1:
        scale = 1/50

        depth_ori = 243#img_density.shape[0]
        height_ori = 440#img_density.shape[1]
        width_ori = 339#img_density.shape[2]

        print(depth_ori)
        print(height_ori)
        print(width_ori)

        img_filename_Nuclear_Isocortex_to_ori = fw_dir_nu + "CUBIC-space_16.7um_grid_points_50um.tif"

        img_N_ori,_ = np.histogramdd(
            np.vstack([
                data_annotated["z"],# 50um scale
                data_annotated["y"], 
                data_annotated["x"] 
                ]).T,
                bins=(depth_ori, height_ori, width_ori),
                range=[(0,depth_ori),(0,height_ori),(0,width_ori)]
            )

        tifffile.imsave(
                img_filename_Nuclear_Isocortex_to_ori,
                img_N_ori.astype(np.float32)
            )
        del img_N_ori
        gc.collect()  
                                                         
    # save split into 6 parts
    if 1==1:
        df_list = []
        df_list = [df_transformed.iloc[offset::6] for offset in range(6)]
        dt_stitched = np.dtype([
            ('z', 'f4'), ('y', 'f4'), ('x', 'f4')
        ])

        for i in [0, 1,2, 3, 4, 5]:
            data_annotated =  np.zeros((len(df_list[i]),), dtype=dt_stitched)
            data_annotated["x"]  = df_list[i]["x"]# 50um scale
            data_annotated["y"]  = df_list[i]["y"]
            data_annotated["z"]  = df_list[i]["z"]
            #data_annotated["id"]  = df_list[i]["id"]
            data_annotated.tofile(fw_dir_nu + "CUBIC-space-grid-16.7um-resolution_ver1_" +str(i)+".bin") 
                                                         
    
    dt_stitched_corrected = np.dtype([
        ('z', 'f4'), ('y', 'f4'), ('x', 'f4'), ('id', 'u4')  # u4 = np.uint32
    ])

    data_annotated =  np.zeros((transformed_points.shape[0],), dtype=dt_stitched_corrected)
    data_annotated["x"]  = transformed_points["x"]# 50um scale
    data_annotated["y"]  = transformed_points["y"]
    data_annotated["z"]  = transformed_points["z"]
    data_annotated["id"]  = transformed_points["id"]
    data_annotated.tofile(fw_dir_nu + "SCA-space-grid_only_Affined-16.7um-resolution_ver1.bin")
    print(data_annotated) 
                                                         
    # rasterize grid points to an image for visual check
    if 1==1:
        scale = 1/50

        depth_ori = 320 #243#img_density.shape[0]
        height_ori = 528 #440#img_density.shape[1]
        width_ori = 456 #339#img_density.shape[2]

        print(depth_ori)
        print(height_ori)
        print(width_ori)

        img_filename_Nuclear_Isocortex_to_ori = fw_dir_nu + "SCA-space_only_Affined_16.7um_grid_points_50um.tif"

        img_N_ori,_ = np.histogramdd(
            np.vstack([
                data_annotated["z"],# 50um scale
                data_annotated["y"], 
                data_annotated["x"] 
                ]).T,
                bins=(depth_ori, height_ori, width_ori),
                range=[(0,depth_ori),(0,height_ori),(0,width_ori)]
            )

        tifffile.imsave(
                img_filename_Nuclear_Isocortex_to_ori,
                img_N_ori.astype(np.float32)
            )
        del img_N_ori
        gc.collect()                      
                                                
    # save split into 6 parts
    if 1==1:
        df_list = []
        df_list = [transformed_points.iloc[offset::6] for offset in range(6)]
        dt_stitched_corrected = np.dtype([
            ('z', 'f4'), ('y', 'f4'), ('x', 'f4'), ('id', 'u4')  # u4 = np.uint32
        ])

        for i in [0, 1,2, 3, 4, 5]:
            data_annotated =  np.zeros((len(df_list[i]),), dtype=dt_stitched_corrected)
            data_annotated["x"]  = df_list[i]["x"]
            data_annotated["y"]  = df_list[i]["y"]
            data_annotated["z"]  = df_list[i]["z"]
            data_annotated["id"]  = df_list[i]["id"]
            data_annotated.tofile(fw_dir_nu + "SCA-space-grid_only_Affined-16.7um-resolution_ver1_" +str(i)+".bin")

                                                         
    end_time = datetime.datetime.now()
    print("End time:", end_time)
    print("Duration:", end_time - start_time)